# Authors
- Raphael Wäspi (18-918-938)
- Nicolas Huber (16-936-205)

In [1]:
import math
# Imports
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import os
import glob
import re
import random
import multiprocessing
from statistics import mean

import scipy.special
from tqdm import tqdm
import shutil
from joblib import Parallel, delayed
from scipy.special import factorial
import powerlaw
from NEMtropy import DirectedGraph, UndirectedGraph
from NEMtropy import matrix_generator as mg
from NEMtropy.network_functions import build_adjacency_from_edgelist


In [2]:
# Define colors with reduced opacity from a colormap
COLOR_BLUE = 'dodgerblue'
COLOR_GREEN = 'palegreen'
COLOR_RED = 'lightcoral'
COLOR_YELLOW = 'gold'
COLOR_GREY = 'slategrey'
COLOR_BLACK = 'black'
COLOR_LIGHT_GRAY = 'lightgray'
COLOR_DARK_GRAY = 'darkgray'

# Define the main data directory
OUTPUT_PATH = "./output"

In [4]:
# Prepare output folder

# Delete the entire 'output' directory and its contents
if os.path.exists(OUTPUT_PATH):
    shutil.rmtree(OUTPUT_PATH)

# Create the 'output' directory
os.makedirs(OUTPUT_PATH)

# Define subdirectories
### OUTPUT_PATH_ERGMS = os.path.join(OUTPUT_PATH, "ERGMs")


# Recreate the subdirectories
### os.makedirs(OUTPUT_PATH_ERGMS)

In [5]:
# Define the directory containing the .gml files
directory = 'datasets'

# Use glob to get a list of all .gml files in the directory
gml_files = glob.glob(os.path.join(directory, '*.gml'))

# Init an empty dictionary for all graphs - the key is the actual name, and the value is the graph data
graphs = {}

# Use tqdm to show a progress bar in Jupyter Notebook
with tqdm(total=len(gml_files), desc="Loading GML Files") as pbar:
    # Loop through the list of .gml files and load them
    for file_path in gml_files:
        with open(file_path, 'r') as file:
            content = file.read()
            
            # Use regex to extract the desired part
            match = re.search(r'/([^/]+)\.gml$', file_path)
            if match:
                graph_name = match.group(1)
                
                # Add to the graph dictionary
                graph = nx.read_gml(file_path)
                graphs[graph_name] = graph
                
                # Print the graph name when each part is finished
                print(f"Finished loading {graph_name}")
                
        pbar.update(1)  # Update the progress bar

Loading GML Files: 100%|██████████| 4/4 [00:00<00:00, 24.43it/s]

Finished loading graph1.2
Finished loading graph_madrid
Finished loading graph_jazz_collab
Finished loading graph1.1


# 1. Compartmental Models on Networks

## 1.1 Compartmental Models on Networks
Write a function to simulate the SI dynamic on a generic network and provide a visualisation.
The function should take in input:
- a network, e.g. if you are using networkx a generic nx.Graph,
- the infection rate β,
- an infected seed node.

You may also want to add other parameters, e.g. the maximum number of steps for the simulation (some
bad things may happen if you don’t). As an output, the function should return the list of nodes infected
in the simulation. Additionally, run the SI function for an adequate number of steps, with β = 0.5 on
the following two graphs (the datasets are provided), with infected seed node “10”:
- graph1.1.gml
- graph1.2.gml

and provide a visualisation of the final state of the epidemic, where all the infected nodes at the end of
the run are red, or blue if they are not infected.